# August 2026 verification · **01 — Bronze**

I land every file unmodified in `data/bronze/<slug>/`, stamped with a SHA-256
and a `metadata.json` (URL, page, vintage, size). Nothing is parsed here.

**Reuse rule for April-July 2026.** Two Maps already downloaded those four
RTT extracts, from the same URLs NHS England still lists. I don't
re-download them. Instead I:

1. check the URL in my catalogue is the one Two Maps recorded;
2. hash the Two Maps file and require it to equal the SHA-256 Two Maps wrote at the time;
3. ask the server for the file's current size and require it to equal the bytes on disk.

If any of the three fails, I download the file fresh and say so.

In [1]:
import json
import shutil
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import (BRONZE_DIR, TWO_MAPS_BRONZE, download, now_utc,  # noqa: E402
                     remote_size, sha256_of, write_metadata)
from sources import SOURCES  # noqa: E402

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

In [2]:
def try_reuse(src: dict, filename: str, url: str, dest: Path) -> str | None:
    """Copy a Two Maps file into my Bronze if, and only if, it is provably the same bytes."""
    tm_folder = TWO_MAPS_BRONZE / src["reuse_from"]
    tm_meta = json.loads((tm_folder / "metadata.json").read_text())
    tm_file = next(f for f in tm_meta["files"] if f["filename"] == filename)
    if tm_file["source_url"] != url:
        return f"URL differs from Two Maps ({tm_file['source_url']})"
    if sha256_of(tm_folder / filename) != tm_file["sha256"]:
        return "Two Maps file no longer matches its own recorded SHA-256"
    live = remote_size(url)
    if live is not None and live != tm_file["bytes"]:
        return f"server now reports {live:,} bytes, Two Maps holds {tm_file['bytes']:,}"
    shutil.copy2(tm_folder / filename, dest)
    return None


log = []
for src in SOURCES:
    folder = BRONZE_DIR / src["slug"]
    folder.mkdir(parents=True, exist_ok=True)
    # On a re-run I keep how each file first arrived, so "reused from Two Maps"
    # is never overwritten by "already in Bronze".
    prev = {}
    if (folder / "metadata.json").exists():
        prev = {f["filename"]: f.get("how") for f in json.loads((folder / "metadata.json").read_text())["files"]}
    files_meta = []
    for filename, url, desc in src["files"]:
        dest = folder / filename
        how = prev.get(filename) or "already in Bronze"
        if not dest.exists():
            how, reason = "downloaded", None
            if src["reuse_from"]:
                reason = try_reuse(src, filename, url, dest)
                how = "reused from Two Maps (hash + size verified)" if reason is None else f"downloaded: {reason}"
            if not dest.exists():
                download(url, dest)
        files_meta.append({"filename": filename, "source_url": url, "description": desc,
                           "bytes": dest.stat().st_size, "sha256": sha256_of(dest), "how": how})
        log.append({"slug": src["slug"], "file": filename, "MB": round(dest.stat().st_size / 1e6, 1), "how": how})
    write_metadata(folder, publisher=src["publisher"], publication_name=src["publication_name"],
                   source_page=src["source_page"], vintage=src["vintage"], grain=src["grain"],
                   primary=src["primary"], files=files_meta)

log = pd.DataFrame(log)
print(f"{len(log)} files, {log.MB.sum():.0f} MB in Bronze")
log

33 files, 180 MB in Bronze


,slug,file,MB,how
0,nhse_rtt_2025-04,Full-CSV-data-file-Apr25-ZIP-4M-revised.zip,3.7,downloaded
1,nhse_rtt_2025-05,Full-CSV-data-file-May25-ZIP-4M-revised.zip,3.6,downloaded
2,nhse_rtt_2025-06,Full-CSV-data-file-Jun25-ZIP-4M-revised.zip,3.7,downloaded
3,nhse_rtt_2025-07,Full-CSV-data-file-Jul25-ZIP-4M-revised-2.zip,3.7,downloaded
4,nhse_rtt_2025-08,Full-CSV-data-file-Aug25-ZIP-4M-revised-2.zip,3.6,downloaded
5,nhse_rtt_2025-09,Full-CSV-data-file-Sep25-ZIP-4M-revised.zip,3.7,downloaded
6,nhse_rtt_2025-10,Full-CSV-data-file-Oct25-ZIP-4M-SrRW6y.zip,3.7,downloaded
7,nhse_rtt_2025-11,Full-CSV-data-file-Nov25-ZIP-4M-1Xmjkk.zip,3.6,downloaded
8,nhse_rtt_2025-12,Full-CSV-data-file-Dec25-ZIP-3M-revised.zip,3.5,downloaded
9,nhse_rtt_2026-01,Full-CSV-data-file-Jan26-ZIP-4M-WL5BiP.zip,3.6,downloaded


## Sanity checks on what landed

A URL that has been rotated can return an HTML error page with status 200.
So I check every payload is the file type its name claims, by its magic
bytes, before Silver touches anything.

In [3]:
MAGIC = {".zip": b"PK\x03\x04", ".xlsx": b"PK\x03\x04", ".pdf": b"%PDF"}
bad = []
for f in sorted(BRONZE_DIR.rglob("*")):
    if f.suffix in MAGIC:
        if f.open("rb").read(4) != MAGIC[f.suffix]:
            bad.append(f.name)
    elif f.suffix == ".csv":
        head = f.open("rb").read(200).decode("utf-8-sig", "replace")
        if head.lstrip().lower().startswith("<"):
            bad.append(f.name)
assert not bad, f"not the file type the name claims: {bad}"
print("every payload has the right magic bytes")

every payload has the right magic bytes
